# Sprint Finish Position - Model Comparison

Compares four approaches for predicting sprint race finish position:
1. Sprint qualifying position as direct proxy (no model)
2. Linear regression with rolling features
3. Main race XGBoost model with sprint qualifying position as `predicted_quali_position`
4. Blend of option 3 and sprint qualifying position

In [10]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import joblib
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

from app.config import (
    PROCESSED_HISTORIC_FEATURES_DIR, PROCESSED_PRACTICE_FEATURES_DIR,
    INTERIM_SPRINT_DIR, INTERIM_SPRINT_QUALIFYING_DIR,
    ARTIFACTS_DIR, TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS
)
from app.models.configs import FINISH_POSITION_MODEL, QUALI_POSITION_MODEL

## Load sprint data

In [11]:
sq_files = sorted(INTERIM_SPRINT_QUALIFYING_DIR.glob('*.parquet'))
sprint_files = sorted(INTERIM_SPRINT_DIR.glob('*.parquet'))

if not sq_files:
    raise FileNotFoundError('No sprint qualifying data found - run ingest-data and clean-data first')

sq = pd.concat([pd.read_parquet(f) for f in sq_files])
sprint = pd.concat([pd.read_parquet(f) for f in sprint_files])

print(f'Sprint qualifying rounds: {sq["race_id"].nunique()}')
print(f'Sprint race rounds: {sprint["race_id"].nunique()}')
sq.head()

Sprint qualifying rounds: 21
Sprint race rounds: 27


,constructor_id,sprint_quali_position,q1_time,q2_time,q3_time,race_id,season,round,driver_id
16,,1.0,102.820,102.500,101.697,2023_04,2023,4,charles_leclerc
11,,2.0,103.858,102.925,101.844,2023_04,2023,4,sergio_perez
1,,3.0,103.288,102.417,101.987,2023_04,2023,4,max_verstappen
63,,4.0,103.763,103.112,102.252,2023_04,2023,4,george_russell
55,,5.0,103.622,102.909,102.287,2023_04,2023,4,carlos_sainz


In [12]:
# join sprint qualifying position onto sprint race results
df = sq[['race_id', 'driver_id', 'sprint_quali_position']].merge(
    sprint[['race_id', 'driver_id', 'finish_position', 'dnf_flag', 'dsq_flag']],
    on=['race_id', 'driver_id']
)

# join historic features
historic = pd.concat([pd.read_parquet(f) for f in sorted(PROCESSED_HISTORIC_FEATURES_DIR.glob('*.parquet'))])
practice = pd.concat([pd.read_parquet(f) for f in sorted(PROCESSED_PRACTICE_FEATURES_DIR.glob('*.parquet'))])

historic = historic.drop(columns=['sprint_quali_position'], errors='ignore')
df = df.merge(historic, on=['race_id', 'driver_id'], how='left')
df = df.merge(practice, on=['race_id', 'driver_id'], how='left')

# season from race_id
df['season'] = df['race_id'].str[:4].astype(int)

df = df.dropna(subset=['finish_position'])

print(f'Total sprint rows: {len(df)}')
print(f'Seasons: {sorted(df["season"].unique())}')
df.head()

Total sprint rows: 425
Seasons: [np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


,race_id,driver_id,sprint_quali_position,finish_position,dnf_flag,dsq_flag,rolling_quali_pos_last_3,rolling_quali_pos_last_5,rolling_finish_pos_last_3,rolling_finish_pos_last_5,...,constructor_rolling_quali_pos_last_3,constructor_form_trend_last_5,fp2_gap_to_leader_pct,fp3_gap_to_leader_pct,fp3_sector1_gap_to_leader_pct,fp3_sector2_gap_to_leader_pct,fp3_sector3_gap_to_leader_pct,fp3_teammate_gap_pct,fp2_longrun_avg_gap_to_field_pct,fp2_laps_completed
0,2023_04,charles_leclerc,1.0,2.0,False,False,4.000000,5.0,15.333333,10.4,...,4.333333,-15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023_04,sergio_perez,2.0,1.0,False,False,7.666667,6.8,2.666667,3.6,...,6.666667,7.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023_04,max_verstappen,3.0,3.0,False,False,5.666667,4.0,1.333333,2.2,...,6.666667,7.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023_04,george_russell,4.0,4.0,False,False,4.000000,4.2,9.666667,7.0,...,5.000000,-10.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023_04,carlos_sainz,5.0,5.0,False,False,4.666667,4.6,7.333333,5.8,...,4.333333,-15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_val  = df[df['season'].isin(VAL_SEASONS)]
df_test = df[df['season'].isin(TEST_SEASONS)]
df_train = df[df['season'].isin(TRAIN_SEASONS)]

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

Train: 119 | Val: 120 | Test: 120


## Evaluate helper

In [14]:
def evaluate(y_true, y_pred, label):
    mask = ~np.isnan(y_pred)
    mae = mean_absolute_error(y_true[mask], y_pred[mask])
    corr = spearmanr(y_true[mask], y_pred[mask]).statistic
    print(f'  [{label}] MAE: {mae:.3f} | Spearman: {corr:.3f}')

## Option 1: Sprint qualifying position as direct proxy

In [15]:
print(df_val['sprint_quali_position'].isna().sum(), 'NaN out of', len(df_val))
print(df_val['race_id'].unique())
print(VAL_SEASONS)
print(df['season'].unique())


0 NaN out of 120
['2024_05' '2024_06' '2024_11' '2024_19' '2024_21' '2024_23']
[2024]
[2023 2024 2025 2026]


In [16]:
print('Option 1: Sprint qualifying position as proxy')
evaluate(df_val['finish_position'].values, df_val['sprint_quali_position'].values, 'val')
evaluate(df_test['finish_position'].values, df_test['sprint_quali_position'].values, 'test')

Option 1: Sprint qualifying position as proxy
  [val] MAE: 2.350 | Spearman: 0.806
  [test] MAE: 3.200 | Spearman: 0.634


## Option 2: Linear regression with rolling features

In [17]:
linear_features = [
    'sprint_quali_position',
    'rolling_finish_pos_last_3',
    'rolling_finish_pos_last_5',
    'rolling_quali_pos_last_3',
    'rolling_quali_pos_last_5',
    'constructor_rolling_finish_pos_last_3',
    'constructor_rolling_finish_pos_last_5',
    'constructor_rolling_quali_pos_last_3',
    'season_points_to_date',
]

X_train_lr = df_train[linear_features]
y_train_lr = df_train['finish_position']

lr = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('model', Ridge())])
lr.fit(X_train_lr, y_train_lr)

print('Option 2: Linear regression')
evaluate(df_val['finish_position'].values, lr.predict(df_val[linear_features]), 'val')
evaluate(df_test['finish_position'].values, lr.predict(df_test[linear_features]), 'test')

Option 2: Linear regression
  [val] MAE: 2.382 | Spearman: 0.781
  [test] MAE: 3.270 | Spearman: 0.629


## Option 3: Main race model with sprint_quali_position as predicted_quali_position

In [18]:
finish_model = joblib.load(ARTIFACTS_DIR / f"{FINISH_POSITION_MODEL['name']}.joblib")

# substitute sprint_quali_position for predicted_quali_position
def predict_sprint_finish(df_split, model, config):
    X = df_split.copy()
    X['predicted_quali_position'] = X['sprint_quali_position']  # add before feature selection
    X = X[config['features']]
    raw = model.predict(X)
    # rank within each race
    ranked = (
        df_split.assign(_pred=raw)
        .groupby('race_id')['_pred']
        .rank(method='first')
        .astype(int)
        .values
    )
    return ranked

val_preds  = predict_sprint_finish(df_val, finish_model, FINISH_POSITION_MODEL)
test_preds = predict_sprint_finish(df_test, finish_model, FINISH_POSITION_MODEL)

print('Option 3: Main race model (sprint_quali_position as predicted_quali_position)')
evaluate(df_val['finish_position'].values, val_preds.astype(float), 'val')
evaluate(df_test['finish_position'].values, test_preds.astype(float), 'test')

Option 3: Main race model (sprint_quali_position as predicted_quali_position)
  [val] MAE: 2.567 | Spearman: 0.771
  [test] MAE: 3.183 | Spearman: 0.666


## Option 4: Blend of option 3 and sprint qualifying position

In [19]:
print('Option 4: Blend (model output + sprint quali position)')
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    val_blend  = w * val_preds  + (1 - w) * df_val['sprint_quali_position'].values
    test_blend = w * test_preds + (1 - w) * df_test['sprint_quali_position'].values
    print(f'  model_weight={w}')
    evaluate(df_val['finish_position'].values, val_blend, 'val')
    evaluate(df_test['finish_position'].values, test_blend, 'test')

Option 4: Blend (model output + sprint quali position)
  model_weight=0.3
  [val] MAE: 2.333 | Spearman: 0.807
  [test] MAE: 3.102 | Spearman: 0.658
  model_weight=0.4
  [val] MAE: 2.347 | Spearman: 0.804
  [test] MAE: 3.093 | Spearman: 0.663
  model_weight=0.5
  [val] MAE: 2.367 | Spearman: 0.801
  [test] MAE: 3.092 | Spearman: 0.666
  model_weight=0.6
  [val] MAE: 2.400 | Spearman: 0.797
  [test] MAE: 3.103 | Spearman: 0.668
  model_weight=0.7
  [val] MAE: 2.435 | Spearman: 0.790
  [test] MAE: 3.117 | Spearman: 0.669


## Summary

In [20]:
results = []

for label, y_true, y_pred in [
    ('1. Sprint quali proxy', df_val['finish_position'].values, df_val['sprint_quali_position'].values),
    ('2. Linear regression',  df_val['finish_position'].values, lr.predict(df_val[linear_features])),
    ('3. Main race model',    df_val['finish_position'].values, val_preds.astype(float)),
]:
    mask = ~np.isnan(y_pred)
    results.append({
        'approach': label,
        'val_mae': mean_absolute_error(y_true[mask], y_pred[mask]),
        'val_spearman': spearmanr(y_true[mask], y_pred[mask]).statistic,
    })

pd.DataFrame(results).set_index('approach')

,val_mae,val_spearman
approach,,
1. Sprint quali proxy,2.350000,0.806015
2. Linear regression,2.381527,0.781296
3. Main race model,2.566667,0.770927
